### Preparação do ambiente analítico e definição da base de entrada

In [4]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

# onde o notebook está rodando
CWD = Path.cwd()
print("CWD:", CWD)

# tenta localizar a raiz do projeto (pasta que contém 'data')
PROJECT_DIR = CWD
for _ in range(6):
    if (PROJECT_DIR / "data").exists():
        break
    PROJECT_DIR = PROJECT_DIR.parent

print("PROJECT_DIR:", PROJECT_DIR)

PARQUET_PATH = PROJECT_DIR / "data" / "processed" / "pgfn" / "pgfn_sida_2024_2025.parquet"
REPORTS_DIR = PROJECT_DIR / "reports"
RUN_DATE = "20260108"

print("PARQUET_PATH:", PARQUET_PATH)
print("EXISTE?", PARQUET_PATH.exists())

pq_file = pq.ParquetFile(PARQUET_PATH)




CWD: d:\pasta_desafio\src
PROJECT_DIR: d:\pasta_desafio
PARQUET_PATH: d:\pasta_desafio\data\processed\pgfn\pgfn_sida_2024_2025.parquet
EXISTE? True


### Validação visual da coerência dos dados consolidados

In [5]:
rg0 = pq_file.read_row_group(0).to_pandas()
head_df = rg0.head(10)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

display(head_df)


,cpf_cnpj,tipo_pessoa,tipo_devedor,nome_devedor,uf_devedor,unidade_responsavel,numero_inscricao,tipo_situacao_inscricao,situacao_inscricao,receita_principal,data_inscricao,indicador_ajuizado,valor_consolidado,ano,trimestre,uf,arquivo_origem
0,XXX735.623XX,Pessoa física,PRINCIPAL,JOSE ALEKSANDRO DA SILVA,AC,ACRE,1010300352142,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - IRPF,25/11/2003,SIM,155196540.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
1,XXX333.272XX,Pessoa física,PRINCIPAL,SANDRA MONICA DAS NEVES GALDINO,DF,ACRE,1011000027007,Benefício Fiscal,ATIVA AJUIZADA NEGOCIADA NO SISPAR,Receita da dívida ativa - IRPF,28/10/2010,SIM,31683.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
2,XXX735.623XX,Pessoa física,PRINCIPAL,JOSE ALEKSANDRO DA SILVA,AC,ACRE,1060400004576,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - CPMF,26/01/2004,SIM,293206.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
3,28.196.878/0001-63,Pessoa jurídica,PRINCIPAL,MORESCHI CLINICA ODONTOLOGICA LTDA,PR,AMAZONAS,2172000069480,Em cobrança,ATIVA EM COBRANCA,Receita da dívida ativa - PIS,11/05/2020,NAO,58234.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
4,XXX156.489XX,Pessoa física,CORRESPONSAVEL,EDUARDO MORESCHI,PR,AMAZONAS,2172000069480,Em cobrança,ATIVA EM COBRANCA,Receita da dívida ativa - PIS,11/05/2020,NAO,58234.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
5,84.461.748/0001-81,Pessoa jurídica,PRINCIPAL,ACBR COMPUTADORES DA AMAZONIA LTDA,SP,AMAZONAS,2179900094488,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - PIS,30/11/1999,SIM,12897724.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
6,XXX273.578XX,Pessoa física,CORRESPONSAVEL,HARRY CHIANG,SP,AMAZONAS,2179900094488,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - PIS,30/11/1999,SIM,12897724.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
7,84.661.206/0001-52,Pessoa jurídica,PRINCIPAL,CAPA CONSTRUCOES R PAVIMENTACAO LTDA,SP,AMAZONAS,2170800004092,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - PIS,14/04/2008,SIM,5550758.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
8,XXX237.772XX,Pessoa física,CORRESPONSAVEL,OTAVIO RAMAN NEVES,AM,AMAZONAS,2170800004092,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - PIS,14/04/2008,SIM,5550758.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv
9,XXX305.023XX,Pessoa física,CORRESPONSAVEL,JORGE HADAD SOBRINHO,MA,AMAZONAS,2170800004092,Em cobrança,ATIVA AJUIZADA,Receita da dívida ativa - PIS,14/04/2008,SIM,5550758.0,2024,1,NA,arquivo_lai_SIDA_1_202403.csv


Esta etapa realiza a validação estrutural inicial da base consolidada em formato Parquet, confirmando a leitura dos dados, o acesso aos registros por meio de amostragem controlada e a consistência do schema. O procedimento garante que o dataset está corretamente organizado e pronto para as análises subsequentes.

### Caracterização estrutural da base PGFN consolidada

In [6]:
pq_file = pq.ParquetFile(PARQUET_PATH)
md = pq_file.metadata

total_rows = md.num_rows
num_row_groups = pq_file.num_row_groups
colnames = pq_file.schema_arrow.names

print("Linhas (metadata):", f"{total_rows:,}")
print("Colunas:", len(colnames))
print("Row groups:", num_row_groups)
colnames


Linhas (metadata): 226,222,615
Colunas: 17
Row groups: 2283


['cpf_cnpj',
 'tipo_pessoa',
 'tipo_devedor',
 'nome_devedor',
 'uf_devedor',
 'unidade_responsavel',
 'numero_inscricao',
 'tipo_situacao_inscricao',
 'situacao_inscricao',
 'receita_principal',
 'data_inscricao',
 'indicador_ajuizado',
 'valor_consolidado',
 'ano',
 'trimestre',
 'uf',
 'arquivo_origem']

A base consolidada possui 226.222.615 registros, caracterizando um dataset de grande porte mesmo após o recorte temporal adotado.
O conjunto apresenta 17 colunas, indicando um schema relativamente enxuto e adequado para análises agregadas e construção de indicadores.
A escrita em 2.283 row groups viabiliza leitura seletiva, estatísticas por bloco e otimização de desempenho, permitindo análises escaláveis sem a necessidade de carregamento integral do dataset em memória.

In [22]:
schema = pq_file.schema_arrow

print("Colunas:", len(schema.names))
for name, field in zip(schema.names, schema):
    print(f"{name:30s} | {field.type}")


Colunas: 18
cpf_cnpj                       | string
tipo_pessoa                    | string
tipo_devedor                   | string
nome_devedor                   | string
uf_unidade_responsavel         | string
unidade_responsavel            | string
numero_inscricao               | string
tipo_situacao_inscricao        | string
situacao_inscricao             | string
receita_principal              | string
data_inscricao                 | string
indicador_ajuizado             | string
valor_consolidado              | double
uf_devedor                     | string
ano                            | int32
trimestre                      | int32
uf                             | string
arquivo_origem                 | string


### Construção de amostra controlada para análise exploratória

In [9]:
MAX_SAMPLE_ROWS = 200_000

parts = [rg0]
count = len(rg0)
rg = 1

while count < MAX_SAMPLE_ROWS and rg < num_row_groups:
    part = pq_file.read_row_group(rg).to_pandas()
    parts.append(part)
    count += len(part)
    rg += 1

sample_df = pd.concat(parts, ignore_index=True).head(MAX_SAMPLE_ROWS)
sample_df.shape


(200000, 17)

Foi construída uma amostra controlada da base consolidada, suficiente para análises estruturais e de qualidade dos dados, sem a necessidade de carregamento completo do dataset.

### Matriz de decisão para limpeza e tipagem baseada em amostra

In [10]:
import pandas as pd
import numpy as np

N = len(sample_df)

rows = []

for col in sample_df.columns:
    s = sample_df[col]

    n_null = s.isna().sum()
    pct_null = round(n_null / N * 100, 2)

    nunique = s.nunique(dropna=True)
    is_constant = nunique == 1
    dtype_atual = str(s.dtype)

    # Heurísticas objetivas de decisão
    if pct_null == 100:
        acao = "remover"
        tipo_destino = "N/A"
        justificativa = "100% nulo na amostra"
    elif is_constant and col not in {"ano", "trimestre", "uf", "arquivo_origem"}:
        acao = "remover"
        tipo_destino = "N/A"
        justificativa = "coluna constante (1 valor)"
    else:
        acao = "manter"

        # sugestão de tipo
        if col == "valor_consolidado":
            tipo_destino = "float64"
            justificativa = "métrica financeira"
        elif col in {"ano", "trimestre"}:
            tipo_destino = "int"
            justificativa = "dimensão temporal"
        elif "data" in col.lower():
            tipo_destino = "datetime"
            justificativa = "variável temporal"
        elif dtype_atual == "object" and nunique <= 200:
            tipo_destino = "category"
            justificativa = "baixa cardinalidade"
        else:
            tipo_destino = "string"
            justificativa = "texto/identificador"

    rows.append({
        "coluna": col,
        "tipo_atual": dtype_atual,
        "tipo_sugerido": tipo_destino,
        "acao_sugerida": acao,
        "nulos_%": pct_null,
        "n_valores_distintos": nunique,
        "constante": is_constant,
        "justificativa": justificativa
    })

decisao_df = (
    pd.DataFrame(rows)
      .sort_values(["acao_sugerida", "nulos_%"], ascending=[True, False])
      .reset_index(drop=True)
)

decisao_df


,coluna,tipo_atual,tipo_sugerido,acao_sugerida,nulos_%,n_valores_distintos,constante,justificativa
0,cpf_cnpj,object,string,manter,0.0,93866,False,texto/identificador
1,tipo_pessoa,object,category,manter,0.0,2,False,baixa cardinalidade
2,tipo_devedor,object,category,manter,0.0,3,False,baixa cardinalidade
3,nome_devedor,object,string,manter,0.0,89277,False,texto/identificador
4,uf_devedor,object,category,manter,0.0,27,False,baixa cardinalidade
5,unidade_responsavel,object,category,manter,0.0,7,False,baixa cardinalidade
6,numero_inscricao,object,string,manter,0.0,145027,False,texto/identificador
7,tipo_situacao_inscricao,object,category,manter,0.0,4,False,baixa cardinalidade
8,situacao_inscricao,object,category,manter,0.0,43,False,baixa cardinalidade
9,receita_principal,object,category,manter,0.0,25,False,baixa cardinalidade


Foi construída uma matriz de decisão automatizada para orientar a limpeza e a tipagem das variáveis com base em métricas objetivas extraídas de uma amostra controlada: taxa de nulos, cardinalidade e constância. Os resultados indicam uma separação clara entre variáveis identificadoras (alta cardinalidade, mantidas como string), variáveis categóricas (baixa cardinalidade, candidatas a category) e variáveis analíticas essenciais (valor_consolidado como numérica e data_inscricao como temporal). Observou-se ainda que ano e trimestre aparecem constantes na amostra, o que é compatível com o recorte temporal adotado. Não foram identificadas colunas 100% nulas na amostra, e as variáveis de localização e origem (uf_devedor, uf e arquivo_origem) permaneceram consistentes para fins de rastreabilidade e segmentação.

In [11]:
from datetime import datetime
RUN_DATE = datetime.now().strftime("%Y%m%d")

decisao_df.to_csv(
    f"../reports/pgfn_tabela_decisao_{RUN_DATE}.csv",
    index=False
)


A matriz de decisão resultante da análise de qualidade e estrutura dos dados foi persistida como um artefato do projeto, garantindo rastreabilidade e transparência no processo de limpeza.

### Validação dos tipos atuais das variáveis

In [7]:
sample_df.dtypes


cpf_cnpj                    object
tipo_pessoa                 object
tipo_devedor                object
nome_devedor                object
uf_unidade_responsavel      object
unidade_responsavel         object
numero_inscricao            object
tipo_situacao_inscricao     object
situacao_inscricao          object
receita_principal           object
data_inscricao              object
indicador_ajuizado          object
valor_consolidado          float64
uf_devedor                  object
ano                          int32
trimestre                    int32
uf                          object
arquivo_origem              object
dtype: object

Esta checagem serve apenas como validação rápida antes das conversões na etapa de limpeza.

### Diagnóstico de valores ausentes na amostra (contagem e percentual)

In [12]:
na_abs = sample_df.isna().sum()
na_pct = (na_abs / len(sample_df) * 100).round(2)

na_tbl = (
    pd.DataFrame({"nulos": na_abs, "pct": na_pct})
      .sort_values("nulos", ascending=False)
)

na_tbl.head(20)


,nulos,pct
cpf_cnpj,0,0.0
tipo_pessoa,0,0.0
tipo_devedor,0,0.0
nome_devedor,0,0.0
uf_devedor,0,0.0
unidade_responsavel,0,0.0
numero_inscricao,0,0.0
tipo_situacao_inscricao,0,0.0
situacao_inscricao,0,0.0
receita_principal,0,0.0


A análise de valores nulos na amostra indica completude total das variáveis avaliadas, não sendo observados registros ausentes em nenhuma coluna neste recorte. Em particular, a variável uf_devedor apresentou preenchimento integral na amostra, sugerindo consistência dos dados neste subconjunto. A avaliação de sua relevância para a ABT deve considerar análises adicionais em outros recortes ou no conjunto completo da base.

In [33]:
import numpy as np
import pandas as pd

# escolher row groups espaçados ao longo do arquivo
rg_indices = np.linspace(
    0,
    num_row_groups - 1,
    num=10,      # quantidade de pontos de verificação
    dtype=int
)

check = []

for rg in rg_indices:
    table = pq_file.read_row_group(rg, columns=["uf_devedor"])
    part = table.to_pandas()

    total = len(part)
    n_null = part["uf_devedor"].isna().sum()

    check.append({
        "row_group": rg,
        "linhas": total,
        "nulos": n_null,
        "pct_nulos": round(n_null / total * 100, 2)
    })

uf_check_df = pd.DataFrame(check)
uf_check_df


,row_group,linhas,nulos,pct_nulos
0,0,48006,48006,100.0
1,320,200000,200000,100.0
2,641,200000,200000,100.0
3,962,200000,200000,100.0
4,1283,200000,0,0.0
5,1603,200000,0,0.0
6,1924,200000,0,0.0
7,2245,200000,0,0.0
8,2566,200000,0,0.0
9,2887,144215,0,0.0


A análise distribuída ao longo da base mostra que a variável uf_devedor apresenta grande quantidade de valores nulos, estando ausente em uma parcela significativa dos dados, além de possuir preenchimento restrito a períodos específicos. Essa baixa representatividade ao longo do conjunto de dados indica que a variável não é elegível para compor a Base Analítica Tratada (ABT).

### Distribuição de valores das variáveis categóricas na amostra


In [13]:
for c in ["uf", "tipo_pessoa", "tipo_devedor", "tipo_situacao_inscricao", "situacao_inscricao", "indicador_ajuizado", "ano", "trimestre"]:
    if c in sample_df.columns:
        display(sample_df[c].value_counts(dropna=False).head(20))


uf
NA    200000
Name: count, dtype: int64

tipo_pessoa
Pessoa jurídica    150969
Pessoa física       49031
Name: count, dtype: int64

tipo_devedor
PRINCIPAL         145027
CORRESPONSAVEL     54407
SOLIDARIO            566
Name: count, dtype: int64

tipo_situacao_inscricao
Em cobrança                      158718
Benefício Fiscal                  39738
Garantia                           1029
Suspenso por decisão judicial       515
Name: count, dtype: int64

situacao_inscricao
ATIVA AJUIZADA                                                            84875
ATIVA EM COBRANCA                                                         48847
ATIVA NAO AJUIZAVEL NEGOCIADA NO SISPAR                                   30698
ATIVA A SER AJUIZADA                                                      19290
ATIVA AJUIZADA NEGOCIADA NO SISPAR                                         7206
ATIVA COM AJUIZAMENTO A SER PROSSEGUIDO                                    3366
ATIVA AJUIZADA PARC LEI 11941/09 ART 1-DIVIDAS SEM PARCEL. ANTERIOR         932
ATIVA COM PARCELAMENTO SIMPLIFICADO RESCINDIDO E AJUIZAM A PROSSEGUIR       694
ATIVA AJUIZADA PARC LEI 11941/09 ART 3-SALDO REMANESCENTE PARCEL            643
ATIVA AJUIZADA EM PROCESSO DE NEGOCIACAO NO SISPAR                          624
ATIVA NAO AJUIZAVEL EM PROCESSO DE NEGOCIACAO NO SISPAR                     551
ATIVA AJUIZADA - GARANTIA - SEGURO GARANTIA                                 391
ATIVA AJUIZADA COM EX

indicador_ajuizado
SIM    100143
NAO     99857
Name: count, dtype: int64

ano
2024    200000
Name: count, dtype: int64

trimestre
1    200000
Name: count, dtype: int64

Os domínios das variáveis categóricas e temporais mostram-se consistentes na amostra, com distribuições coerentes com o contexto da dívida ativa e sem indícios de inconsistências estruturais. Observa-se que as variáveis ano e trimestre permanecem constantes, o que é compatível com o recorte temporal adotado. Destaca-se ainda que a variável uf apresenta valor único (NA) em toda a amostra, indicando ausência de preenchimento neste recorte específico, aspecto que deve ser considerado na definição final das variáveis da ABT.